In [1]:
import pandas as pd
import numpy as np
import matplotlib as plt
import sklearn

**Crypto price prediction depends on two main factors: historical time series data and real-world market sentiment. Unlike stock price prediction, where price history often carries most of the signal, crypto is strongly influenced by traders' reactions to political, economic and financial events worldwide. Our dataset provides only the price side, namely open, high, low, close and daily variation, so we must separately obtain a sentiment score for each day in the historical window we are studying. Merging these daily scores with the OHLC data by date gives the model both price behavior and market mood as inputs.**

In [2]:
pd.options.display.float_format = '{:.2f}'.format
pd.options.display.max_rows = 100
pd.options.display.max_columns = 25

In [3]:
# Load data from source

data = pd.read_csv(r"C:\Users\ACER\OneDrive\Desktop\ML Notes and colab Notebooks\Data Science Projects\Cryptoprice-prediction\notebook\data\eda.csv")

In [4]:
df = data.copy()

In [5]:
df

,high,volume,marketCap,variation,crypto_name,date
0,777.51,46862700.00,8955394563.50,8.96,Bitcoin,2013-12-27
1,25.27,31112200.00,566088038.79,13.50,Litecoin,2013-12-27
2,0.00,477422.00,8016603.65,26.41,Dogecoin,2013-12-27
3,0.03,148422.00,211674056.85,15.28,XRP,2013-12-27
4,0.03,143404.00,213453408.57,5.55,XRP,2013-12-28
...,...,...,...,...,...,...
69247,0.02,40401338.08,1652957186.63,3.56,VeChain,2022-10-23
69248,1.53,28443510.43,1572824923.08,6.62,Flow,2022-10-23
69249,5.15,106949683.62,1559551358.49,4.11,Filecoin,2022-10-23
69250,0.00,214326817.63,1576291167.45,7.54,Terra Classic,2022-10-23


In [6]:
df['crypto_name'].value_counts()


crypto_name
Bitcoin                  3012
Litecoin                 3012
Dogecoin                 3012
XRP                      3011
Monero                   2866
Stellar                  2791
Ethereum                 2424
Ethereum Classic         2072
Tether                   1994
Basic Attention Token    1760
EOS                      1728
BNB                      1706
Bitcoin Cash             1699
Chainlink                1649
Decentraland             1641
TRON                     1641
Cardano                  1637
Maker                    1530
Theta Network            1524
Ravencoin                1469
Huobi Token              1431
Tezos                    1362
VeChain                  1332
Quant                    1274
USD Coin                 1257
Cronos                   1196
Cosmos                   1062
Polygon                  1054
UNUS SED LEO             1012
Chiliz                    983
FTX Token                 961
OKB                       956
Algorand                  95

# Feature Engeenering

In [42]:
df = df.drop(columns=["Unnamed: 0",'timestamp'], axis = 1)

In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72946 entries, 0 to 72945
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   open         72946 non-null  float64
 1   high         72946 non-null  float64
 2   low          72946 non-null  float64
 3   close        72946 non-null  float64
 4   volume       72946 non-null  float64
 5   marketCap    72946 non-null  float64
 6   crypto_name  72946 non-null  object 
 7   date         72946 non-null  object 
dtypes: float64(6), object(2)
memory usage: 4.5+ MB


In [44]:
df.date = df.date.str.replace('-','/')
df.date = pd.to_datetime(df['date'])


In [45]:
#To see actual crypto names 
#df["crypto_name"].astype('category').cat.categories

In [46]:
import xgboost as xgb
from catboost import CatBoostClassifier 
from sklearn.preprocessing import StandardScaler


num_df = df.select_dtypes(include= 'number')
cat_df = df.select_dtypes(include= ['object','datetime64'])
num_scaled = StandardScaler().fit_transform(num_df)
scaled_num_df = pd.DataFrame(
    num_scaled,
    columns=num_df.columns,
    index=num_df.index
)
sclaed_df = pd.concat([scaled_num_df,cat_df], axis = 1)




In [47]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)

_df = pd.DataFrame(scaled_num_df,columns= scaled_num_df.columns, index= scaled_num_df.index)
pca_array = pca.fit_transform(_df)
pca_df = pd.DataFrame(pca_array)


In [48]:
print(type(pca_df))

<class 'pandas.core.frame.DataFrame'>


In [49]:
transformed_df = pd.concat([cat_df,pca_df], axis = 1)

In [51]:
transformed_df.rename(columns={0:'PC1',1:'PC2'})

,crypto_name,date,PC1,PC2
0,Bitcoin,2013-05-05,-0.37,-0.18
1,Litecoin,2013-05-05,-0.42,-0.17
2,Bitcoin,2013-05-06,-0.37,-0.18
3,Litecoin,2013-05-06,-0.42,-0.17
4,Bitcoin,2013-05-07,-0.37,-0.18
...,...,...,...,...
72941,VeChain,2022-10-23,-0.41,-0.16
72942,Flow,2022-10-23,-0.41,-0.16
72943,Filecoin,2022-10-23,-0.41,-0.15
72944,Terra Classic,2022-10-23,-0.40,-0.14
